In [2]:
import pandas as pd
import numpy as np

df=pd.read_csv("data.csv")

print(df)


df.info()
df.isnull().sum()


                                CONS_NO  FLAG  2014/1/1  2014/1/10  2014/1/11  \
0      0387DD8A07E07FDA6271170F86AD9151     1       NaN        NaN        NaN   
1      01D6177B5D4FFE0CABA9EF17DAFC2B84     1       NaN        NaN        NaN   
2      4B75AC4F2D8434CFF62DB64D0BB43103     1       NaN        NaN        NaN   
3      B32AC8CC6D5D805AC053557AB05F5343     1       NaN        NaN        NaN   
4      EDFC78B07BA2908B3395C4EB2304665E     1      2.90       3.42       3.81   
...                                 ...   ...       ...        ...        ...   
42367  F1472871E1AFF49D4289564B6377D76C     0       NaN        NaN        NaN   
42368  F3C8BBCD2DC26C1E0249DEEF6A4256B7     0      2.70       4.39       3.95   
42369  A9A0FE83467A680FBFB0DBFC910DF227     0      0.58       0.84       1.61   
42370  D9A6ADA018FA46A55D5438370456AA45     0     16.89      13.84      13.50   
42371  F3406636BAD1E6E0826E8EDDC9A1BF00     0       NaN        NaN        NaN   

       2014/1/12  2014/1/13

CONS_NO          0
FLAG             0
2014/1/1     16502
2014/1/10    16460
2014/1/11    16461
             ...  
2016/9/5       499
2016/9/6       522
2016/9/7       405
2016/9/8       328
2016/9/9       345
Length: 1036, dtype: int64

In [4]:
df['CONS_NO'] = "HOUSE " + (df.index + 1).astype(str)
print(df.head())



   CONS_NO  FLAG  2014/1/1  2014/1/10  2014/1/11  2014/1/12  2014/1/13  \
0  HOUSE 1     1       NaN        NaN        NaN        NaN        NaN   
1  HOUSE 2     1       NaN        NaN        NaN        NaN        NaN   
2  HOUSE 3     1       NaN        NaN        NaN        NaN        NaN   
3  HOUSE 4     1       NaN        NaN        NaN        NaN        NaN   
4  HOUSE 5     1       2.9       3.42       3.81       4.58       3.56   

   2014/1/14  2014/1/15  2014/1/16  ...  2016/9/28  2016/9/29  2016/9/3  \
0        NaN        NaN        NaN  ...      10.12       9.96     16.92   
1        NaN        NaN        NaN  ...       0.00       0.00      0.00   
2        NaN        NaN        NaN  ...        NaN        NaN       NaN   
3        NaN        NaN        NaN  ...       6.50       9.99     11.78   
4       4.25       3.86       3.53  ...      17.77      10.37     15.32   

   2016/9/30  2016/9/4  2016/9/5  2016/9/6  2016/9/7  2016/9/8  2016/9/9  
0       7.60     27.22     18

In [6]:
import pandas as pd
import numpy as np

# --- 1. Data Load and Setup (Dummy data for demonstration) ---
# REAL DATA MEIN: df_original = pd.read_csv('your_sgcc_file.csv')


# Separate FLAG aur CONS_NO ko index set karna
df_flag = df[['CONS_NO', 'FLAG']].set_index('CONS_NO')
df_consumption_raw = df.drop('FLAG', axis=1).set_index('CONS_NO')


# --- 2. Data Cleaning aur Missing Data Feature (NaNs) ---

# missing_day_count (Original NaN count) ko pehle calculate karein
df_features = pd.DataFrame(index=df_consumption_raw.index)
df_features['missing_day_count'] = df_consumption_raw.isnull().sum(axis=1) # Row-wise sum

# Missing Values ko handle karein: fillna(0)
df_clean = df_consumption_raw.fillna(0)

# Outliers ko handle karein: Capping (Bohot high values ko limit karna)
MAX_DAILY_CONSUMPTION = 500.0
df_clean = df_clean.clip(upper=MAX_DAILY_CONSUMPTION, axis=1)


# --- 3. Feature Engineering (Calculation Row-wise: axis=1) ---

# A. Statistical Features
df_features['avg_consumption'] = df_clean.mean(axis=1)
df_features['std_consumption'] = df_clean.std(axis=1)
df_features['min_consumption'] = df_clean.min(axis=1)
df_features['max_consumption'] = df_clean.max(axis=1)
df_features['total_consumption'] = df_clean.sum(axis=1)

# B. Anomaly Features

# zero_consumption_days
# Count karein kitne din consumption bilkul 0 thi
df_features['zero_consumption_days'] = (df_clean == 0).sum(axis=1)

# max_to_mean_ratio
# Division by zero (jab avg_consumption 0 ho) se bachne ke liye epsilon (choti value) use karein
epsilon = 1e-6
df_features['max_to_mean_ratio'] = df_features['max_consumption'] / (df_features['avg_consumption'] + epsilon)


# --- 4. Final Data Setup ---

# Features aur Target variable (FLAG) ko join karna
df_model_ready = pd.merge(df_features, df_flag, left_index=True, right_index=True)

# Final Model-Ready Data ka CSV
df_model_ready.to_csv('sgcc_engineered_features.csv')

In [7]:
df_model_ready.head()


,missing_day_count,avg_consumption,std_consumption,min_consumption,max_consumption,total_consumption,zero_consumption_days,max_to_mean_ratio,FLAG
CONS_NO,,,,,,,,,
HOUSE 1,443,7.943830,9.592727,0.0,46.49,8213.92,443,5.852340,1
HOUSE 2,730,0.000000,0.000000,0.0,0.00,0.00,1034,0.000000,1
HOUSE 3,775,2.485145,5.488486,0.0,32.81,2569.64,816,13.202443,1
HOUSE 4,562,6.670106,10.607305,0.0,52.76,6896.89,562,7.909918,1
HOUSE 5,4,8.217824,5.952928,0.0,43.43,8497.23,4,5.284853,1
